# Simulation du tri clinique : seuils réglés localement et alertes de sécurité (notebook 15)

**Pourquoi ce notebook ?** L'objectif 6 du sujet demande une application clinique. Le notebook 14 a montré que le décalage entre la validation et le test n'abîme pas la capacité à *classer* les radios (l'AUC de chaque modèle reste entre 0,89 et 0,94) mais rend le *seuil de décision* inutilisable : à un seuil de 0, la spécificité tombe à 32–62 % sur le test. Un outil utile doit donc (1) ne pas décider seul, (2) régler ses seuils sur des radios du centre qui l'utilise, (3) savoir dire quand il sort de son domaine de validité. Ce notebook simule cet outil sur les scores déjà calculés (aucun réseau n'est relancé) et fabrique le fichier de paramètres de l'application.

## Le tri en trois zones

Chaque radio reçoit un score : la moyenne, sur les trois graines, de logit PNEUMONIA − logit NORMAL. Deux seuils, réglés sur un échantillon étiqueté, découpent l'échelle :

| Zone | Règle | Conduite proposée |
|---|---|---|
| **Verte** | score < seuil bas. Le seuil bas est fixé pour que **au moins 98 %** des radios PNEUMONIA de l'échantillon de réglage aient un score au-dessus | lecture en file standard |
| **Orange** | entre les deux seuils | lecture médicale sans priorité particulière |
| **Rouge** | score > seuil haut. Le seuil haut est fixé pour que **au moins 90 %** des radios NORMAL de l'échantillon de réglage aient un score en dessous | lecture prioritaire |

**Règle de sécurité.** Si une alerte se déclenche (format inhabituel, résolution, luminosité, contraste, bandes noires, poumons non localisés en v4, désaccord entre les trois graines), une radio classée verte passe en orange : une image qui sort du cadre validé n'est jamais « rassurante ». Les zones ne sont pas des diagnostics.

Les fonctions de tri, de réglage des seuils et d'alerte viennent de `outils_clinique.py`, le même fichier que celui de l'application : la simulation et l'outil appliquent exactement les mêmes règles.

## Ce que le notebook mesure

1. **Décalage validation → test** : AUC, sensibilité et spécificité à un seuil de 0, et déplacement des scores de chaque classe.
2. **Seuils du développement** (réglés sur la validation) appliqués au test : part des PNEUMONIA en zone verte, part des NORMAL en zone rouge, répartition des radios.
3. **Alertes** : plages réglées sur l'apprentissage, part des radios alertées dans chaque ensemble.
4. **Calibration locale répétée** : le test est coupé en deux moitiés de patients (aucun patient dans les deux), 1 000 fois. Les seuils sont réglés sur une moitié (échantillon local de 25 %, 50 % ou 100 % de cette moitié) et mesurés sur l'autre. Cela montre ce qu'apporte une calibration locale et combien de radios elle demande.
5. **Prévalences** : les mesures précédentes sont indépendantes de la prévalence. On les convertit en charge de travail et en valeur prédictive de la zone rouge et de la zone verte pour des prévalences de 5, 10, 20, 40 % et celle du test.
6. **Efficacité des alertes** : part des erreurs graves (PNEUMONIA en zone verte ou NORMAL en zone rouge) qui tombent sur des radios alertées, effet de la règle de sécurité.
7. **Fichier de paramètres de l'application** (`tri_parametres.json`).

## Statut : exploratoire, sans verdict

Le test a déjà servi au notebook 14 et son décalage inspire ce notebook. Rien ici ne change les verdicts H1 à H4 et rien ne prouve qu'un centre réel se comportera comme la moitié « évaluation » d'un test tiré du même jeu de données : les deux moitiés viennent du même centre et de la même période. La calibration locale simulée ici est **optimiste** par rapport à un vrai déploiement. Une validation prospective, dans le centre visé, reste nécessaire (chapitre Application clinique du mémoire).

## Attentes écrites le 29/09/2026, avant les résultats de ce notebook

Elles s'appuient sur ce que le notebook 14 avait déjà montré (AUC conservée, spécificité effondrée à un seuil de 0). Elles sont comparées aux résultats sans verdict formel.

| | Attente |
|---|---|
| A1 | Avec les seuils du développement, plus de 20 % des radios NORMAL du test tombent en zone rouge (cible : 10 %). |
| A2 | Avec la calibration locale complète (moitié du test), la médiane de la part de NORMAL en zone rouge sur l'autre moitié est entre 7 et 14 %, et la médiane de la part de PNEUMONIA en zone verte est entre 1 et 5 %. |
| A3 | Un échantillon de réglage plus petit élargit l'intervalle de ces deux mesures, surtout celle des PNEUMONIA en zone verte. |
| A4 | Le taux d'erreurs graves est plus élevé parmi les radios alertées que parmi les autres. |

**Limites connues avant les résultats :** un seul jeu de données ; 618 radios de test, patients regroupés ; moitiés tirées dans le même centre ; le seuil de 98 % de sensibilité en zone verte repose, dans un échantillon de réglage de 100 à 200 PNEUMONIA, sur les 2 à 4 radios les moins bien notées ; les alertes sont réglées sur l'apprentissage (plages de 1 % à 99 %) et n'ont jamais été testées sur des images d'un autre centre.

**Exécution :** noyau `memoire_cnn`, cellules dans l'ordre, sans GPU. **Environ 10 à 20 minutes** (la moitié est la lecture de 5 371 images pour calculer leurs statistiques d'alerte, une seule fois). Prérequis : le dossier `application_clinique` (fichiers `outils_clinique.py` et `application_clinique.py`) est copié dans le dossier du projet. Le notebook 16 est facultatif : s'il est terminé, ses modèles v4 sont ajoutés à la simulation. À la fin, un seul fichier est à transmettre : `tri_clinique/a_envoyer_notebook_15.zip` ; le notebook copie aussi `tri_parametres.json` dans `application_clinique/`.

In [ ]:
from pathlib import Path
from importlib import metadata
import hashlib
import json
import sys
import time
import zipfile
import numpy as np
import pandas as pd
import PIL
from PIL import Image
import scipy
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
DOSSIER_08 = PROJET / "optimisation" / "robustesse_validation_v2"
DOSSIER_10 = PROJET / "optimisation" / "robustesse_augmentation_v2"
DOSSIER_12 = PROJET / "optimisation" / "v3_sans_raccourci_format"
DOSSIER_14 = PROJET / "evaluations" / "test_final_v2_v3"
DOSSIER_16 = PROJET / "optimisation" / "poumons_v4"
APPLICATION = PROJET / "application_clinique"
SORTIE = PROJET / "optimisation" / "tri_clinique"

exiger_presence = [(APPLICATION / "outils_clinique.py", "Copier le dossier application_clinique (outils_clinique.py et application_clinique.py) dans le dossier du projet.")]
for fichier, message in exiger_presence:
    if not fichier.is_file():
        raise RuntimeError(f"{fichier} est introuvable. {message}")
sys.path.insert(0, str(APPLICATION))
import outils_clinique as oc

GRAINES = oc.GRAINES
ARCHITECTURES_NOMS = ["CNN_classique", "CNN_CBAM"]
LIBELLES_MODELES = {"CNN_classique": "CNN classique", "CNN_CBAM": "CNN + CBAM"}
V2_SANS, V2_AVEC, V3, V4 = "v2 sans augmentation", "v2 avec augmentation", "v3 sans raccourci", "v4 recadrage poumons"
assert V3 == oc.V3 and V4 == oc.V4
PREFIXES = {V2_SANS: "v2_sans", V2_AVEC: "v2_avec", V3: "v3", V4: "v4"}
N_REPETITIONS, GRAINE_TRI = 1000, 20260929
FRACTIONS_CALIBRATION = [0.25, 0.5, 1.0]
PREVALENCES = [0.05, 0.10, 0.20, 0.40]
COULEURS = {V2_SANS: "#2a78d6", V2_AVEC: "#eb6834", V3: "#1baf7a", V4: "#b8428c"}
ENCRE, ENCRE_DISCRETE = "#0b0b0b", "#898781"
ZONES_LIBELLES = ["verte", "orange", "rouge"]
ZONES_COULEURS = ["#2f9e6f", "#f2a93b", "#d64545"]

def version_de(bibliotheque):
    try:
        return metadata.version(bibliotheque)
    except metadata.PackageNotFoundError:
        return "absente"

VERSIONS = {"python": sys.version, "numpy": np.__version__, "pandas": pd.__version__, "pillow": PIL.__version__, "scipy": scipy.__version__,
            "matplotlib": version_de("matplotlib")}

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def empreinte_objet(objet):
    return hashlib.sha256(json.dumps(objet, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

def remplacer(temporaire, chemin, essais=20, attente=1.5):
    # Windows ou OneDrive peuvent verrouiller un fichier quelques secondes : on réessaie.
    for essai in range(essais):
        try:
            temporaire.replace(chemin)
            return
        except PermissionError:
            if essai == essais - 1:
                raise PermissionError(f"Impossible d'écrire {chemin} : fermer ce fichier (Excel ?) "
                                      "ou suspendre OneDrive, puis relancer la cellule.")
            time.sleep(attente)

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(objet, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    remplacer(temporaire, chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g")
    remplacer(temporaire, chemin)

def enregistrer_figure(figure, chemin):
    temporaire = chemin.with_suffix(".tmp")
    figure.savefig(temporaire, format="png", dpi=160, bbox_inches="tight")
    remplacer(temporaire, chemin)

def styliser(axe):
    axe.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    axe.set_axisbelow(True)
    for cote in ["top", "right"]:
        axe.spines[cote].set_visible(False)

for dossier in [SORTIE, SORTIE / "figures"]:
    dossier.mkdir(parents=True, exist_ok=True)
print("PyTorch non nécessaire : ce notebook lit les scores déjà enregistrés.")
print("Résultats dans :", SORTIE)

In [ ]:
# Répartition (notebook 02) et scores déjà enregistrés par les notebooks 08, 10, 12, 14 et, si terminé, 16.
lecture = {"dtype": {"groupe": str, "empreinte": str}}
train_df = pd.read_csv(REPARTITION / "train.csv", **lecture)
val_df = pd.read_csv(REPARTITION / "val.csv", **lecture)
test_df = pd.read_csv(REPARTITION / "test.csv", **lecture)
for nom_ensemble, tableau in [("train", train_df), ("val", val_df), ("test", test_df)]:
    exiger({"chemin_relatif", "groupe", "classe"}.issubset(tableau.columns), f"Colonnes absentes : {nom_ensemble}.")
    exiger(tableau["chemin_relatif"].is_unique, f"Chemins dupliqués : {nom_ensemble}.")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ensembles = {"train": train_df, "val": val_df, "test": test_df}
    exiger(set(ensembles[a]["groupe"]).isdisjoint(ensembles[b]["groupe"]), f"{a} et {b} partagent des patients.")
CLASSES = oc.CLASSES
Y_VAL = val_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
Y_TEST = test_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
N_VAL, N_TEST = len(val_df), len(test_df)
IDENTIFIANTS_VAL, IDENTIFIANTS_TEST = val_df["chemin_relatif"].tolist(), test_df["chemin_relatif"].tolist()
# Un patient n'a qu'une classe : condition du tirage par patient (même hypothèse qu'au notebook 14).
for nom_ensemble, tableau in [("val", val_df), ("test", test_df)]:
    exiger((tableau.groupby("groupe")["classe"].nunique() == 1).all(), f"Un patient a des radios de deux classes : {nom_ensemble}.")

def fichier_validation(version, nom, graine):
    if version == V2_SANS:
        return DOSSIER_08 / "predictions" / f"{nom}_graine_{graine}__originale.csv"
    if version == V2_AVEC:
        return DOSSIER_10 / "predictions" / f"avec__{nom}_graine_{graine}__originale.csv"
    if version == V3:
        return DOSSIER_12 / "predictions_validation" / f"{nom}_graine_{graine}.csv"
    return DOSSIER_16 / "predictions_validation" / f"{PREFIXES[V4]}__{nom}_graine_{graine}.csv"

def fichier_test(version, nom, graine):
    if version == V4:
        return DOSSIER_16 / "predictions_test" / f"{PREFIXES[V4]}__{nom}_graine_{graine}.csv"
    return DOSSIER_14 / "predictions" / f"{PREFIXES[version]}__{nom}_graine_{graine}.csv"

def lire_scores(chemin, identifiants, score_pneumonie_est_difference_logits=False):
    # Le score est toujours recalculé à partir des deux logits. La colonne score_pneumonie n'a pas le même sens partout :
    # probabilité softmax dans les fichiers des notebooks 08 et 10, absente des fichiers du notebook 12, différence des
    # logits dans les fichiers de test des notebooks 14 et 16. Elle n'est donc contrôlée que dans ce dernier cas.
    exiger(chemin.is_file(), f"Fichier de scores introuvable : {chemin}")
    tableau = pd.read_csv(chemin, dtype={"groupe": str}, float_precision="round_trip")
    exiger(tableau["chemin_relatif"].tolist() == identifiants, f"Ordre des radios différent : {chemin}")
    exiger({"logit_NORMAL", "logit_PNEUMONIA"}.issubset(tableau.columns), f"Colonnes des logits absentes : {chemin}")
    exiger(np.isfinite(tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=float)).all(), f"Scores non finis : {chemin}")
    score = tableau["logit_PNEUMONIA"].to_numpy(dtype=np.float64) - tableau["logit_NORMAL"].to_numpy(dtype=np.float64)
    if score_pneumonie_est_difference_logits:
        exiger("score_pneumonie" in tableau.columns, f"Colonne score_pneumonie absente : {chemin}")
        exiger(np.allclose(score, tableau["score_pneumonie"].to_numpy(dtype=np.float64), rtol=0, atol=1e-9), f"score_pneumonie incohérent : {chemin}")
    return score

VERSIONS_SIMULEES = [V2_SANS, V2_AVEC, V3]
V4_DISPONIBLE = (DOSSIER_16 / "termine.json").is_file()
if V4_DISPONIBLE:
    VERSIONS_SIMULEES.append(V4)
SYSTEMES = [(v, m) for v in VERSIONS_SIMULEES for m in ARCHITECTURES_NOMS]
SCORES_VAL, SCORES_TEST = {}, {}
for version, nom in SYSTEMES:
    SCORES_VAL[(version, nom)] = np.stack([lire_scores(fichier_validation(version, nom, g), IDENTIFIANTS_VAL) for g in GRAINES], axis=1)
    SCORES_TEST[(version, nom)] = np.stack([lire_scores(fichier_test(version, nom, g), IDENTIFIANTS_TEST, score_pneumonie_est_difference_logits=True)
                                            for g in GRAINES], axis=1)
    if version != V4:
        # Contrôle : les prédictions du test portent bien les étiquettes du test.
        for g in GRAINES:
            classe_vraie = pd.read_csv(fichier_test(version, nom, g))["classe_vraie"].to_numpy(dtype=np.int64)
            exiger((classe_vraie == Y_TEST).all(), f"Étiquettes du test différentes : {fichier_test(version, nom, g)}")
ENSEMBLE_VAL = {cle: v.mean(axis=1) for cle, v in SCORES_VAL.items()}      # score d'un système = moyenne sur les 3 graines
ENSEMBLE_TEST = {cle: v.mean(axis=1) for cle, v in SCORES_TEST.items()}
PRINCIPAUX = [(V3, m) for m in ARCHITECTURES_NOMS] + ([(V4, m) for m in ARCHITECTURES_NOMS] if V4_DISPONIBLE else [])

def libelle(cle):
    return f"{LIBELLES_MODELES[cle[1]]}, {cle[0]}"

print(f"{len(SYSTEMES)} systèmes ({'v4 inclus' if V4_DISPONIBLE else 'v4 absent : notebook 16 non terminé'}) ; "
      f"validation {N_VAL} radios, test {N_TEST} radios ({int((Y_TEST == 0).sum())} NORMAL, {int((Y_TEST == 1).sum())} PNEUMONIA).")
print("Score d'un système = moyenne sur les 3 graines de logit PNEUMONIA − logit NORMAL.")

## 1. Le décalage entre validation et test : le classement tient, le seuil non

Pour chaque système (score moyenné sur les trois graines) : AUC sur la validation et sur le test, sensibilité et spécificité à un seuil de 0 (le seuil par défaut du réseau), et déplacement de la médiane des scores de chaque classe. La dernière colonne donne le seuil qui rendrait au test la spécificité de la validation : c'est le décalage à corriger, en unités de score.

In [ ]:
def mesures_seuil(y, score, seuil=0.0):
    prediction = score > seuil
    return {"sensibilite": float(prediction[y == 1].mean()), "specificite": float((~prediction[y == 0]).mean())}

lignes = []
for cle in SYSTEMES:
    sv, st = ENSEMBLE_VAL[cle], ENSEMBLE_TEST[cle]
    mv, mt = mesures_seuil(Y_VAL, sv), mesures_seuil(Y_TEST, st)
    seuil_equivalent = float(np.quantile(st[Y_TEST == 0], mv["specificite"], method="higher"))
    lignes.append({"version": cle[0], "modele": cle[1], "auc_validation": roc_auc_score(Y_VAL, sv), "auc_test": roc_auc_score(Y_TEST, st),
                   "sensibilite_validation_seuil0": mv["sensibilite"], "sensibilite_test_seuil0": mt["sensibilite"],
                   "specificite_validation_seuil0": mv["specificite"], "specificite_test_seuil0": mt["specificite"],
                   "mediane_NORMAL_validation": float(np.median(sv[Y_VAL == 0])), "mediane_NORMAL_test": float(np.median(st[Y_TEST == 0])),
                   "mediane_PNEUMONIA_validation": float(np.median(sv[Y_VAL == 1])), "mediane_PNEUMONIA_test": float(np.median(st[Y_TEST == 1])),
                   "seuil_qui_rendrait_la_specificite_de_validation_au_test": seuil_equivalent})
DECALAGE = pd.DataFrame(lignes)
ecrire_csv(DECALAGE, SORTIE / "decalage_validation_test.csv")
print("Score moyen sur 3 graines. Sensibilité = part des PNEUMONIA au-dessus de 0 ; spécificité = part des NORMAL en dessous de 0.")
display(DECALAGE.round(3))

fig, axes = plt.subplots(1, len(PRINCIPAUX), figsize=(4.6 * len(PRINCIPAUX), 4.4), squeeze=False, constrained_layout=True)
for axe, cle in zip(axes[0], PRINCIPAUX):
    for classe, nom_classe, style in [(0, "NORMAL", "-"), (1, "PNEUMONIA", "-")]:
        for ensemble, y, scores, trait, epaisseur in [("validation", Y_VAL, ENSEMBLE_VAL[cle], "--", 1.3), ("test", Y_TEST, ENSEMBLE_TEST[cle], "-", 2.0)]:
            s = np.sort(scores[y == classe])
            axe.plot(s, np.arange(1, len(s) + 1) / len(s), trait, linewidth=epaisseur, color=["#2a78d6", "#d64545"][classe],
                     label=f"{nom_classe}, {ensemble}")
    axe.axvline(0, color=ENCRE_DISCRETE, linewidth=1)
    axe.set_title(libelle(cle), fontsize=11)
    axe.set_xlabel("Score (logit PNEUMONIA − logit NORMAL)")
    axe.set_ylabel("Part des radios de la classe dont le score est ≤ x")
    styliser(axe)
axes[0, 0].legend(fontsize=8, frameon=False, loc="lower right")
fig.suptitle("Fonctions de répartition des scores par classe : validation (tirets) et test (trait plein). Ligne verticale : seuil 0")
enregistrer_figure(fig, SORTIE / "figures" / "decalage_scores_validation_test.png")
plt.show()
plt.close(fig)

## 2. Les seuils du développement, appliqués au test

Les deux seuils sont réglés sur la **validation** (cibles : 98 % de PNEUMONIA au-dessus du seuil bas, 90 % de NORMAL sous le seuil haut), puis appliqués tels quels au test. Le tableau donne, sur la validation (là où ils ont été réglés) puis sur le test : part des PNEUMONIA en zone verte (les cas manqués), part des NORMAL en zone rouge (fausses alarmes), et répartition de toutes les radios.

In [ ]:
def taux_zones(y, score, bas, haut):
    zone = oc.zone_de(score, bas, haut)
    ligne = {}
    for classe, nom_classe in [(1, "PNEUMONIA"), (0, "NORMAL")]:
        for k, z in enumerate(ZONES_LIBELLES):
            ligne[f"{z}_parmi_{nom_classe}_pct"] = 100 * float((zone[y == classe] == k).mean())
    for k, z in enumerate(ZONES_LIBELLES):
        ligne[f"part_{z}_toutes_pct"] = 100 * float((zone == k).mean())
    return ligne

SEUILS_DEV = {}
lignes = []
for cle in SYSTEMES:
    sv = ENSEMBLE_VAL[cle]
    seuils = oc.seuils_depuis_calibration(sv[Y_VAL == 0], sv[Y_VAL == 1])
    SEUILS_DEV[cle] = seuils
    for ensemble, y, scores in [("validation (réglage)", Y_VAL, sv), ("test", Y_TEST, ENSEMBLE_TEST[cle])]:
        lignes.append({"version": cle[0], "modele": cle[1], "ensemble": ensemble, "seuil_bas": seuils["seuil_bas"], "seuil_haut": seuils["seuil_haut"],
                       "zones_chevauchantes": seuils["zones_chevauchantes"], **taux_zones(y, scores, seuils["seuil_bas"], seuils["seuil_haut"])})
DEVELOPPEMENT_SUR_TEST = pd.DataFrame(lignes)
ecrire_csv(DEVELOPPEMENT_SUR_TEST, SORTIE / "seuils_developpement_sur_test.csv")
print("Seuils réglés sur la validation. Cibles : PNEUMONIA en zone verte ≈ 2 % ; NORMAL en zone rouge ≈ 10 %.")
display(DEVELOPPEMENT_SUR_TEST[["version", "modele", "ensemble", "seuil_bas", "seuil_haut", "verte_parmi_PNEUMONIA_pct", "rouge_parmi_NORMAL_pct",
                                "verte_parmi_NORMAL_pct", "rouge_parmi_PNEUMONIA_pct", "part_verte_toutes_pct", "part_orange_toutes_pct",
                                "part_rouge_toutes_pct"]].round(2))

## 3. Alertes de sécurité : réglage sur l'apprentissage, contrôle sur la validation et le test

Sept alertes existent dans l'application. Cinq ne dépendent que de l'image : **format** (proportions hors de la plage des radios d'apprentissage), **résolution** (très petit côté), **luminosité**, **contraste** et **bandes noires**. Leurs plages sont les quantiles 1 % et 99 % de l'apprentissage. Elles ne se déclenchent donc que sur environ 1 à 2 % des radios d'apprentissage par construction. Deux dépendent du modèle : **désaccord entre les trois graines** (écart-type des trois scores au-dessus du 90ᵉ centile de la validation) et **poumons non localisés** (v4 seulement, masque invalide au notebook 16).

Les statistiques de 5 371 radios sont calculées ici une seule fois avec la fonction de l'application (`statistiques_alerte`) et gardées dans `statistiques_images.csv`.

In [ ]:
fichier_stats = SORTIE / "statistiques_images.csv"
TOUTES_IMAGES = pd.concat([t.assign(ensemble=e)[["ensemble", "chemin_relatif", "groupe", "classe"]]
                           for e, t in [("train", train_df), ("val", val_df), ("test", test_df)]], ignore_index=True)
STATISTIQUES = None
if fichier_stats.exists():
    lu = pd.read_csv(fichier_stats, dtype={"groupe": str})
    if lu["chemin_relatif"].tolist() == TOUTES_IMAGES["chemin_relatif"].tolist():
        STATISTIQUES = lu
        print("Statistiques des images relues :", fichier_stats.name)
if STATISTIQUES is None:
    lignes = []
    for ligne in tqdm(TOUTES_IMAGES.to_dict("records"), desc="Statistiques des radios"):
        with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
            lignes.append({**ligne, **oc.statistiques_alerte(image)})
    STATISTIQUES = pd.DataFrame(lignes)
    ecrire_csv(STATISTIQUES, fichier_stats)
STAT_TRAIN = STATISTIQUES[STATISTIQUES["ensemble"] == "train"]
PARAMETRES_ALERTES = oc.parametres_alertes_depuis_apprentissage(STAT_TRAIN)
print("Plages des alertes (apprentissage, quantiles 1 % et 99 %) :")
display(pd.DataFrame([PARAMETRES_ALERTES]).round(3).T.rename(columns={0: "valeur"}))

CODES_IMAGE = ["format", "resolution", "luminosite", "contraste", "bandes_noires"]
def alertes_par_image(tableau):
    drapeaux = {code: np.zeros(len(tableau), dtype=bool) for code in CODES_IMAGE}
    for i, ligne in enumerate(tableau.to_dict("records")):
        for code in oc.alertes_image(ligne, PARAMETRES_ALERTES):
            drapeaux[code][i] = True
    return pd.DataFrame(drapeaux)

DRAPEAUX_IMAGE = {e: alertes_par_image(STATISTIQUES[STATISTIQUES["ensemble"] == e].reset_index(drop=True)) for e in ["train", "val", "test"]}
for e, ids in [("val", IDENTIFIANTS_VAL), ("test", IDENTIFIANTS_TEST)]:
    exiger(STATISTIQUES[STATISTIQUES["ensemble"] == e]["chemin_relatif"].tolist() == ids, f"Ordre des statistiques différent : {e}.")

if V4_DISPONIBLE:
    qualite = pd.read_csv(DOSSIER_16 / "qualite_masques.csv", dtype={"groupe": str}).set_index("chemin_relatif")
    MASQUE_VALIDE = {"val": qualite.loc[IDENTIFIANTS_VAL, "masque_valide"].astype(bool).to_numpy(),
                     "test": qualite.loc[IDENTIFIANTS_TEST, "masque_valide"].astype(bool).to_numpy()}

SEUIL_GRAINES, DRAPEAUX_SYSTEME = {}, {}
for cle in SYSTEMES:
    SEUIL_GRAINES[cle] = float(np.quantile(oc.ecart_type_graines(SCORES_VAL[cle]), 0.90))
    for e, scores in [("val", SCORES_VAL[cle]), ("test", SCORES_TEST[cle])]:
        graines = oc.ecart_type_graines(scores) > SEUIL_GRAINES[cle]
        poumons = ~MASQUE_VALIDE[e] if cle[0] == V4 else np.zeros(len(scores), dtype=bool)
        image = DRAPEAUX_IMAGE[e].to_numpy(dtype=bool).any(axis=1)
        DRAPEAUX_SYSTEME[(cle, e)] = {"graines": graines, "poumons": poumons, "image": image, "toutes": graines | poumons | image}

lignes = []
for e in ["train", "val", "test"]:
    ligne = {"ensemble": e, "radios": len(DRAPEAUX_IMAGE[e])}
    ligne.update({f"alerte_{code}_pct": 100 * float(DRAPEAUX_IMAGE[e][code].mean()) for code in CODES_IMAGE})
    ligne["au_moins_une_alerte_image_pct"] = 100 * float(DRAPEAUX_IMAGE[e].to_numpy(dtype=bool).any(axis=1).mean())
    lignes.append(ligne)
TAUX_ALERTES_IMAGE = pd.DataFrame(lignes)
lignes = []
for cle in SYSTEMES:
    for e in ["val", "test"]:
        d = DRAPEAUX_SYSTEME[(cle, e)]
        lignes.append({"version": cle[0], "modele": cle[1], "ensemble": e, "seuil_ecart_type_graines": SEUIL_GRAINES[cle],
                       "alerte_graines_pct": 100 * float(d["graines"].mean()), "alerte_poumons_pct": 100 * float(d["poumons"].mean()),
                       "au_moins_une_alerte_pct": 100 * float(d["toutes"].mean())})
TAUX_ALERTES_SYSTEME = pd.DataFrame(lignes)
ecrire_csv(TAUX_ALERTES_IMAGE, SORTIE / "alertes_taux_images.csv")
ecrire_csv(TAUX_ALERTES_SYSTEME, SORTIE / "alertes_taux_systemes.csv")
print("Part des radios qui déclenchent chaque alerte (pourcentage) :")
display(TAUX_ALERTES_IMAGE.round(1))
display(TAUX_ALERTES_SYSTEME.round(1))

## 4. Calibration locale simulée, répétée 1 000 fois

Le test est coupé en deux **moitiés de patients** (chaque patient et toutes ses radios dans une seule moitié ; classes réparties séparément). Les seuils sont réglés sur la moitié « calibration », avec 25 %, 50 % ou 100 % de ses patients (les patients sont pris dans un ordre aléatoire, les petits échantillons sont donc emboîtés dans les grands). Ils sont mesurés sur l'autre moitié. Le tirage est refait 1 000 fois (graine fixée).

À chaque tirage, pour chaque système : part des PNEUMONIA en zone verte (cas manqués), part des NORMAL en zone rouge (fausses alarmes), répartition complète des radios avant et après la règle de sécurité, et, pour l'échantillon de réglage complet, le nombre d'erreurs graves parmi les radios alertées et non alertées. **Erreur grave** : radio PNEUMONIA classée verte, ou radio NORMAL classée rouge, avant la règle de sécurité.

In [ ]:
groupes_test = test_df["groupe"].to_numpy()
PATIENTS = {c: [] for c in [0, 1]}
for g in pd.unique(groupes_test):
    indices = np.flatnonzero(groupes_test == g)
    PATIENTS[int(Y_TEST[indices[0]])].append(indices)

def pourcentages_zones(zone_normal, zone_pneumonie):
    ligne = {}
    for classe, z in [("NORMAL", zone_normal), ("PNEUMONIA", zone_pneumonie)]:
        for k, nom_zone in enumerate(ZONES_LIBELLES):
            ligne[f"{nom_zone}_parmi_{classe}_pct"] = 100 * float((z == k).mean())
    return ligne

rng = np.random.Generator(np.random.PCG64(GRAINE_TRI))
lignes = []
for repetition in tqdm(range(N_REPETITIONS), desc="Calibrations locales simulées"):
    calibration, evaluation = {}, {}
    for c in [0, 1]:
        ordre = rng.permutation(len(PATIENTS[c]))
        moitie = int(np.ceil(len(ordre) / 2))
        calibration[c] = [PATIENTS[c][k] for k in ordre[:moitie]]
        evaluation[c] = [PATIENTS[c][k] for k in ordre[moitie:]]
    idx_eval = {c: np.concatenate(evaluation[c]) for c in [0, 1]}
    for fraction in FRACTIONS_CALIBRATION:
        idx_cal = {c: np.concatenate(calibration[c][:max(1, int(np.ceil(fraction * len(calibration[c]))))]) for c in [0, 1]}
        for cle in SYSTEMES:
            s = ENSEMBLE_TEST[cle]
            seuils = oc.seuils_depuis_calibration(s[idx_cal[0]], s[idx_cal[1]])
            bas, haut = seuils["seuil_bas"], seuils["seuil_haut"]
            z_n, z_p = oc.zone_de(s[idx_eval[0]], bas, haut), oc.zone_de(s[idx_eval[1]], bas, haut)
            alerte = DRAPEAUX_SYSTEME[(cle, "test")]["toutes"]
            a_n, a_p = alerte[idx_eval[0]], alerte[idx_eval[1]]
            zf_n, zf_p = oc.zone_finale(z_n, a_n), oc.zone_finale(z_p, a_p)
            ligne = {"version": cle[0], "modele": cle[1], "fraction": fraction, "repetition": repetition,
                     "n_calibration_NORMAL": len(idx_cal[0]), "n_calibration_PNEUMONIA": len(idx_cal[1]),
                     "seuil_bas": bas, "seuil_haut": haut, "zones_chevauchantes": seuils["zones_chevauchantes"],
                     **pourcentages_zones(z_n, z_p),
                     **{f"finale_{k}": v for k, v in pourcentages_zones(zf_n, zf_p).items()},
                     "n_evaluation": len(idx_eval[0]) + len(idx_eval[1]),
                     "n_alertees": int(a_n.sum() + a_p.sum()),
                     "graves_alertees": int(((z_p == 0) & a_p).sum() + ((z_n == 2) & a_n).sum()),
                     "graves_non_alertees": int(((z_p == 0) & ~a_p).sum() + ((z_n == 2) & ~a_n).sum())}
            lignes.append(ligne)
REPETITIONS = pd.DataFrame(lignes)
exiger(len(REPETITIONS) == N_REPETITIONS * len(FRACTIONS_CALIBRATION) * len(SYSTEMES), "Répétitions incomplètes.")
print(f"{len(REPETITIONS)} calibrations simulées ({N_REPETITIONS} tirages × {len(FRACTIONS_CALIBRATION)} tailles × {len(SYSTEMES)} systèmes).")

In [ ]:
def p2_5(s):
    return float(np.quantile(s, 0.025))

def p97_5(s):
    return float(np.quantile(s, 0.975))

MESURES_CALIBRATION = ["verte_parmi_PNEUMONIA_pct", "rouge_parmi_NORMAL_pct", "verte_parmi_NORMAL_pct", "rouge_parmi_PNEUMONIA_pct",
                       "orange_parmi_PNEUMONIA_pct", "orange_parmi_NORMAL_pct", "finale_verte_parmi_PNEUMONIA_pct", "finale_verte_parmi_NORMAL_pct",
                       "seuil_bas", "seuil_haut", "n_calibration_NORMAL", "n_calibration_PNEUMONIA"]
groupes_synthese = REPETITIONS.groupby(["version", "modele", "fraction"], sort=False)
RESUME_CALIBRATION = groupes_synthese[MESURES_CALIBRATION].agg(["median", p2_5, p97_5])
RESUME_CALIBRATION.columns = [f"{mesure}_{statistique}" for mesure, statistique in RESUME_CALIBRATION.columns]
RESUME_CALIBRATION = RESUME_CALIBRATION.reset_index()
RESUME_CALIBRATION["part_tirages_zones_chevauchantes_pct"] = groupes_synthese["zones_chevauchantes"].mean().to_numpy() * 100
# Lignes rangées par système puis par taille de l'échantillon de réglage (lecture plus facile).
ORDRE_SYSTEMES = {cle: k for k, cle in enumerate(SYSTEMES)}
RESUME_CALIBRATION = (RESUME_CALIBRATION.assign(ordre_systeme=[ORDRE_SYSTEMES[(v, m)] for v, m in zip(RESUME_CALIBRATION["version"], RESUME_CALIBRATION["modele"])])
                      .sort_values(["ordre_systeme", "fraction"], kind="stable").drop(columns="ordre_systeme").reset_index(drop=True))
ecrire_csv(RESUME_CALIBRATION, SORTIE / "calibration_locale_resume.csv")
print("Calibration locale simulée (médiane [2,5 % ; 97,5 %] sur 1 000 tirages ; mesures sur la moitié non utilisée pour le réglage) :")
apercu = RESUME_CALIBRATION[["version", "modele", "n_calibration_NORMAL_median", "n_calibration_PNEUMONIA_median"]].copy()
apercu.insert(2, "part de la moitié utilisée pour le réglage", [f"{100 * f:.0f} %" for f in RESUME_CALIBRATION["fraction"]])
for mesure, nom_colonne in [("verte_parmi_PNEUMONIA_pct", "PNEUMONIA en zone verte (%)"), ("rouge_parmi_NORMAL_pct", "NORMAL en zone rouge (%)")]:
    apercu[nom_colonne] = [f"{m:.1f} [{a:.1f} ; {b:.1f}]" for m, a, b in zip(RESUME_CALIBRATION[f"{mesure}_median"],
                                                                     RESUME_CALIBRATION[f"{mesure}_p2_5"], RESUME_CALIBRATION[f"{mesure}_p97_5"])]
display(apercu.round(1))

# Figure : cas manqués et fausses alarmes selon la taille de l'échantillon de réglage, systèmes principaux.
fig, axes = plt.subplots(2, len(PRINCIPAUX), figsize=(4.6 * len(PRINCIPAUX), 7.6), squeeze=False, constrained_layout=True)
for colonne, cle in enumerate(PRINCIPAUX):
    for rang, (mesure, cible, titre_y) in enumerate([("verte_parmi_PNEUMONIA_pct", 100 * (1 - oc.CIBLE_SENSIBILITE_VERTE), "PNEUMONIA en zone verte (%)"),
                                                     ("rouge_parmi_NORMAL_pct", 100 * (1 - oc.CIBLE_SPECIFICITE_ROUGE), "NORMAL en zone rouge (%)")]):
        axe = axes[rang, colonne]
        t = RESUME_CALIBRATION[(RESUME_CALIBRATION["version"] == cle[0]) & (RESUME_CALIBRATION["modele"] == cle[1])].reset_index(drop=True)
        x = np.arange(len(t)) + 1
        axe.vlines(x, t[f"{mesure}_p2_5"], t[f"{mesure}_p97_5"], color=COULEURS[cle[0]], linewidth=6, alpha=0.35)
        axe.scatter(x, t[f"{mesure}_median"], s=70, color=COULEURS[cle[0]], edgecolors="white", linewidths=1.5, zorder=3, label="calibration locale (médiane, 95 % des tirages)")
        dev = DEVELOPPEMENT_SUR_TEST[(DEVELOPPEMENT_SUR_TEST["version"] == cle[0]) & (DEVELOPPEMENT_SUR_TEST["modele"] == cle[1])
                                     & (DEVELOPPEMENT_SUR_TEST["ensemble"] == "test")].iloc[0][mesure]
        axe.scatter([0], [dev], s=90, marker="D", color=ENCRE, zorder=3, label="seuils du développement, appliqués au test")
        axe.axhline(cible, color=ENCRE_DISCRETE, linewidth=1, linestyle="--", label="cible")
        axe.set_xticks(range(0, len(t) + 1), ["développement"] + [f"{int(round(a))} NORMAL\n{int(round(b))} PNEUMONIA" for a, b in
                                                                   zip(t["n_calibration_NORMAL_median"], t["n_calibration_PNEUMONIA_median"])], fontsize=8)
        axe.set_xlim(-0.5, len(t) + 0.5)
        axe.set_ylim(bottom=0)
        axe.set_ylabel(titre_y)
        if rang == 0:
            axe.set_title(libelle(cle), fontsize=11)
        styliser(axe)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Ce que la calibration locale change (test coupé en deux moitiés de patients ; 1 000 tirages). Axe horizontal : radios de réglage")
enregistrer_figure(fig, SORTIE / "figures" / "calibration_locale.png")
plt.show()
plt.close(fig)

## 5. Charge de travail et valeur des zones selon la prévalence

Les parts de PNEUMONIA en zone verte et de NORMAL en zone rouge ne dépendent pas de la prévalence (proportion de pneumonies parmi les radios reçues). En revanche, ce que le médecin voit en pratique en dépend : la part de radios en zone rouge, la proportion de vraies pneumonies parmi elles (valeur prédictive de la zone rouge) et la proportion de vraies radios normales parmi les radios vertes (valeur prédictive de la zone verte). Le tableau utilise la moyenne, sur les 1 000 tirages, des taux de la calibration locale complète, **après la règle de sécurité** (la moyenne garde des zones dont la somme fait 100 %), pour des prévalences de 5, 10, 20 et 40 % et pour celle du test. La prévalence du centre visé est inconnue : c'est une donnée à mesurer avant tout déploiement.

In [ ]:
complet = REPETITIONS[REPETITIONS["fraction"] == 1.0]
# Moyenne sur les 1 000 tirages : contrairement aux médianes, les moyennes des trois zones font toujours 100 %.
taux_finaux = complet.groupby(["version", "modele"], sort=False)[[f"finale_{z}_parmi_{c}_pct" for z in ZONES_LIBELLES for c in ["NORMAL", "PNEUMONIA"]]].mean()
PREVALENCE_TEST = float(Y_TEST.mean())
lignes = []
for (version, nom), t in taux_finaux.iterrows():
    for prevalence in PREVALENCES + [PREVALENCE_TEST]:
        p = {z: t[f"finale_{z}_parmi_PNEUMONIA_pct"] / 100 for z in ZONES_LIBELLES}
        n = {z: t[f"finale_{z}_parmi_NORMAL_pct"] / 100 for z in ZONES_LIBELLES}
        parts = {z: prevalence * p[z] + (1 - prevalence) * n[z] for z in ZONES_LIBELLES}
        lignes.append({"version": version, "modele": nom, "prevalence_pct": 100 * prevalence,
                       "part_verte_pct": 100 * parts["verte"], "part_orange_pct": 100 * parts["orange"], "part_rouge_pct": 100 * parts["rouge"],
                       "valeur_predictive_zone_rouge_pct": 100 * prevalence * p["rouge"] / parts["rouge"] if parts["rouge"] > 0 else np.nan,
                       "valeur_predictive_zone_verte_pct": 100 * (1 - prevalence) * n["verte"] / parts["verte"] if parts["verte"] > 0 else np.nan,
                       "pneumonies_en_zone_verte_pour_1000_radios": 1000 * prevalence * p["verte"],
                       "fausses_alarmes_rouges_pour_1000_radios": 1000 * (1 - prevalence) * n["rouge"]})
SCENARIOS_PREVALENCE = pd.DataFrame(lignes)
ecrire_csv(SCENARIOS_PREVALENCE, SORTIE / "prevalence_scenarios.csv")
print(f"Prévalence du test : {100 * PREVALENCE_TEST:.1f} % de PNEUMONIA. Taux de la calibration locale complète (moyenne sur les tirages), après règle de sécurité :")
est_principal = [(v, m) in PRINCIPAUX for v, m in zip(SCENARIOS_PREVALENCE["version"], SCENARIOS_PREVALENCE["modele"])]
display(SCENARIOS_PREVALENCE[est_principal].round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
grille = np.linspace(0.01, 0.95, 95)
for cle in PRINCIPAUX:
    t = taux_finaux.loc[cle]
    p_rouge, n_rouge = t["finale_rouge_parmi_PNEUMONIA_pct"] / 100, t["finale_rouge_parmi_NORMAL_pct"] / 100
    p_vert, n_vert = t["finale_verte_parmi_PNEUMONIA_pct"] / 100, t["finale_verte_parmi_NORMAL_pct"] / 100
    style = "-" if cle[1] == "CNN_CBAM" else "--"
    axes[0].plot(100 * grille, 100 * grille * p_rouge / (grille * p_rouge + (1 - grille) * n_rouge), style, color=COULEURS[cle[0]], linewidth=2, label=libelle(cle))
    axes[1].plot(100 * grille, 100 * (1 - grille) * n_vert / (grille * p_vert + (1 - grille) * n_vert), style, color=COULEURS[cle[0]], linewidth=2, label=libelle(cle))
axes[0].set_title("Valeur prédictive de la zone rouge :\npart de vraies PNEUMONIA parmi les radios rouges", fontsize=11)
axes[1].set_title("Valeur prédictive de la zone verte :\npart de vraies radios NORMAL parmi les radios vertes", fontsize=11)
for axe in axes:
    axe.axvline(100 * PREVALENCE_TEST, color=ENCRE_DISCRETE, linewidth=1, linestyle=":")
    axe.set_xlabel("Prévalence de la pneumonie parmi les radios reçues (%)\npointillé : prévalence du test")
    axe.set_ylabel("%")
    axe.set_ylim(0, 100)
    styliser(axe)
axes[0].legend(fontsize=8, frameon=False, loc="lower right")
fig.suptitle("Ce que voit le médecin selon la prévalence (taux de la calibration locale complète, après règle de sécurité)")
enregistrer_figure(fig, SORTIE / "figures" / "prevalence.png")
plt.show()
plt.close(fig)

## 6. Les alertes repèrent-elles les erreurs graves ? Effet de la règle de sécurité

Avec l'échantillon de réglage complet, sur les radios de la moitié « évaluation » : part des radios alertées, nombre d'erreurs graves pour 100 radios alertées et pour 100 radios non alertées, part de toutes les erreurs graves qui tombent sur des radios alertées, et effet de la règle « verte + alerte → orange » sur la part de PNEUMONIA en zone verte (bénéfice) et de NORMAL en zone verte (coût : moins de radios rassurantes).

In [ ]:
lignes = []
for (version, nom), t in complet.groupby(["version", "modele"], sort=False):
    total, alertees = t["n_evaluation"].sum(), t["n_alertees"].sum()
    graves_a, graves_n = t["graves_alertees"].sum(), t["graves_non_alertees"].sum()
    lignes.append({"version": version, "modele": nom, "part_alertees_pct": 100 * alertees / total,
                   "erreurs_graves_pour_100_alertees": 100 * graves_a / alertees if alertees else np.nan,
                   "erreurs_graves_pour_100_non_alertees": 100 * graves_n / (total - alertees),
                   "part_des_erreurs_graves_sur_alertees_pct": 100 * graves_a / (graves_a + graves_n) if graves_a + graves_n else np.nan,
                   "PNEUMONIA_en_zone_verte_sans_regle_pct": float(t["verte_parmi_PNEUMONIA_pct"].median()),
                   "PNEUMONIA_en_zone_verte_avec_regle_pct": float(t["finale_verte_parmi_PNEUMONIA_pct"].median()),
                   "NORMAL_en_zone_verte_sans_regle_pct": float(t["verte_parmi_NORMAL_pct"].median()),
                   "NORMAL_en_zone_verte_avec_regle_pct": float(t["finale_verte_parmi_NORMAL_pct"].median())})
EFFET_ALERTES = pd.DataFrame(lignes)
ecrire_csv(EFFET_ALERTES, SORTIE / "alertes_effet.csv")
print("Alertes et règle de sécurité (échantillon de réglage complet ; somme sur 1 000 tirages ; médianes pour les zones) :")
display(EFFET_ALERTES.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
positions = np.arange(len(PRINCIPAUX))
for axe, (colonne_a, colonne_b, titre) in zip(axes, [("erreurs_graves_pour_100_alertees", "erreurs_graves_pour_100_non_alertees", "Erreurs graves pour 100 radios"),
                                                      ("NORMAL_en_zone_verte_sans_regle_pct", "NORMAL_en_zone_verte_avec_regle_pct", "NORMAL en zone verte (%)")]):
    valeurs_a = [EFFET_ALERTES[(EFFET_ALERTES["version"] == c[0]) & (EFFET_ALERTES["modele"] == c[1])][colonne_a].iat[0] for c in PRINCIPAUX]
    valeurs_b = [EFFET_ALERTES[(EFFET_ALERTES["version"] == c[0]) & (EFFET_ALERTES["modele"] == c[1])][colonne_b].iat[0] for c in PRINCIPAUX]
    axe.bar(positions - 0.2, valeurs_a, 0.38, color=[COULEURS[c[0]] for c in PRINCIPAUX], edgecolor=ENCRE, linewidth=0.8,
            label="radios alertées" if colonne_a.startswith("erreurs") else "sans la règle de sécurité")
    axe.bar(positions + 0.2, valeurs_b, 0.38, color=[COULEURS[c[0]] for c in PRINCIPAUX], edgecolor=ENCRE, linewidth=0.8, alpha=0.45, hatch="//",
            label="radios non alertées" if colonne_a.startswith("erreurs") else "avec la règle de sécurité")
    axe.set_xticks(positions, [libelle(c).replace(", ", "\n") for c in PRINCIPAUX], fontsize=8)
    axe.set_title(titre, fontsize=11)
    axe.legend(fontsize=8, frameon=False)
    styliser(axe)
fig.suptitle("Les alertes concentrent-elles les erreurs graves ? Et que coûte la règle de sécurité ?")
enregistrer_figure(fig, SORTIE / "figures" / "alertes.png")
plt.show()
plt.close(fig)

# Attentes écrites avant les résultats, avec les valeurs observées (sans verdict formel).
lignes = []
for cle in PRINCIPAUX:
    dev = DEVELOPPEMENT_SUR_TEST[(DEVELOPPEMENT_SUR_TEST["version"] == cle[0]) & (DEVELOPPEMENT_SUR_TEST["modele"] == cle[1])
                                 & (DEVELOPPEMENT_SUR_TEST["ensemble"] == "test")].iloc[0]
    complet_cle = RESUME_CALIBRATION[(RESUME_CALIBRATION["version"] == cle[0]) & (RESUME_CALIBRATION["modele"] == cle[1]) & (RESUME_CALIBRATION["fraction"] == 1.0)].iloc[0]
    petit = RESUME_CALIBRATION[(RESUME_CALIBRATION["version"] == cle[0]) & (RESUME_CALIBRATION["modele"] == cle[1]) & (RESUME_CALIBRATION["fraction"] == 0.25)].iloc[0]
    effet = EFFET_ALERTES[(EFFET_ALERTES["version"] == cle[0]) & (EFFET_ALERTES["modele"] == cle[1])].iloc[0]
    lignes.append({"systeme": libelle(cle),
                   "A1_NORMAL_en_rouge_seuils_developpement_pct (attente > 20)": dev["rouge_parmi_NORMAL_pct"],
                   "A2_NORMAL_en_rouge_calibration_locale_mediane_pct (attente 7 à 14)": complet_cle["rouge_parmi_NORMAL_pct_median"],
                   "A2_PNEUMONIA_en_vert_calibration_locale_mediane_pct (attente 1 à 5)": complet_cle["verte_parmi_PNEUMONIA_pct_median"],
                   "A3_largeur_intervalle_PNEUMONIA_en_vert_25pct_vs_100pct (points)": f"{petit['verte_parmi_PNEUMONIA_pct_p97_5'] - petit['verte_parmi_PNEUMONIA_pct_p2_5']:.1f} vs "
                                                                                     f"{complet_cle['verte_parmi_PNEUMONIA_pct_p97_5'] - complet_cle['verte_parmi_PNEUMONIA_pct_p2_5']:.1f}",
                   "A4_erreurs_graves_pour_100_alertees_vs_non_alertees": f"{effet['erreurs_graves_pour_100_alertees']:.1f} vs {effet['erreurs_graves_pour_100_non_alertees']:.1f}"})
ATTENTES = pd.DataFrame(lignes)
ecrire_csv(ATTENTES, SORTIE / "attentes_et_observations.csv")
print("Attentes écrites avant les résultats et valeurs observées (sans verdict formel) :")
display(ATTENTES.T)

## 7. Fichier de paramètres pour l'application

Le fichier `tri_parametres.json` réunit, pour les systèmes que l'application sait charger (v3, et v4 si le notebook 16 est terminé) : les seuils du développement (réglés sur la validation), le seuil du désaccord entre graines, les plages des alertes d'image et, à titre d'exemple seulement, les seuils qu'on obtiendrait en prenant tout le test comme échantillon local. **L'application n'utilise que les seuils du développement, et le dit à l'écran, tant qu'aucune calibration locale n'a été faite** (commande `calibrer` de l'application, sur des radios étiquetées du centre visé).

In [ ]:
import datetime as dt
SYSTEMES_APPLICATION = [cle for cle in SYSTEMES if cle[0] in (V3, V4)]
lignes_json = {}
for cle in SYSTEMES_APPLICATION:
    st = ENSEMBLE_TEST[cle]
    lignes_json[f"{cle[0]}|{cle[1]}"] = {
        "seuils_developpement": SEUILS_DEV[cle],
        "seuil_ecart_graines": SEUIL_GRAINES[cle],
        "auc_validation": float(roc_auc_score(Y_VAL, ENSEMBLE_VAL[cle])), "auc_test": float(roc_auc_score(Y_TEST, st)),
        "seuils_exemple_test_entier": oc.seuils_depuis_calibration(st[Y_TEST == 0], st[Y_TEST == 1]),
        "remarque": "seuils_exemple_test_entier : illustration seulement (le test entier joue le rôle d'un échantillon local) ; non utilisé par l'application."}
PROTOCOLE = {"version_analyse": "tri_clinique_v1", "cibles": {"sensibilite_verte": oc.CIBLE_SENSIBILITE_VERTE, "specificite_rouge": oc.CIBLE_SPECIFICITE_ROUGE},
             "repetitions": N_REPETITIONS, "graine": GRAINE_TRI, "fractions": FRACTIONS_CALIBRATION, "prevalences": PREVALENCES,
             "systemes": [list(c) for c in SYSTEMES], "v4_inclus": V4_DISPONIBLE, "versions": VERSIONS,
             "sha256_csv": {e: empreinte_fichier(REPARTITION / f"{e}.csv") for e in ["train", "val", "test"]},
             "attentes": "A1 à A4 écrites le 29/09/2026 avant les résultats (voir le notebook)"}
ID_PROTOCOLE = empreinte_objet(PROTOCOLE)
PARAMETRES_TRI = {"version": "tri_parametres_v1", "id_protocole": ID_PROTOCOLE, "date": dt.datetime.now().isoformat(timespec="seconds"),
                  "cibles": PROTOCOLE["cibles"],
                  "regle_de_securite": "une alerte empêche la zone verte : verte + alerte → orange",
                  "alertes": {"parametres_images": PARAMETRES_ALERTES, "desaccord_graines": "écart-type des 3 scores > 90e centile de la validation (par système)"},
                  "systemes": lignes_json,
                  "avertissement": oc.AVERTISSEMENT}
ecrire_json(PARAMETRES_TRI, SORTIE / "tri_parametres.json")
APPLICATION.mkdir(parents=True, exist_ok=True)
ecrire_json(PARAMETRES_TRI, APPLICATION / "tri_parametres.json")
print("Paramètres de l'application écrits :", APPLICATION / "tri_parametres.json")
print("Systèmes disponibles dans l'application :", list(lignes_json))

ecrire_json({"id_protocole": ID_PROTOCOLE, "v4_inclus": V4_DISPONIBLE, "systemes": len(SYSTEMES), "repetitions": N_REPETITIONS}, SORTIE / "termine.json")
FICHIERS_A_ENVOYER = ["decalage_validation_test.csv", "seuils_developpement_sur_test.csv", "alertes_taux_images.csv", "alertes_taux_systemes.csv",
                      "calibration_locale_resume.csv", "prevalence_scenarios.csv", "alertes_effet.csv", "attentes_et_observations.csv",
                      "tri_parametres.json", "termine.json"]
archive = SORTIE / "a_envoyer_notebook_15.zip"
temporaire = archive.with_suffix(".tmp")
with zipfile.ZipFile(temporaire, "w", compression=zipfile.ZIP_DEFLATED) as fichier_zip:
    for nom_tableau in FICHIERS_A_ENVOYER:
        fichier_zip.write(SORTIE / nom_tableau, arcname=nom_tableau)
    for figure in sorted((SORTIE / "figures").glob("*.png")):
        fichier_zip.write(figure, arcname=f"figures/{figure.name}")
remplacer(temporaire, archive)
print("Terminé. Pour transmettre les résultats, un seul fichier suffit :")
print(" ", archive)

## Lecture des résultats

**Décalage validation → test.** Regarder d'abord la figure `decalage_scores_validation_test.png` : si les courbes des NORMAL du test sont décalées vers la droite (scores plus élevés) par rapport à celles de la validation tout en restant séparées de celles des PNEUMONIA, le modèle classe toujours bien mais son seuil est faux : c'est l'argument central pour une calibration locale. Si les deux courbes de chaque classe se ressemblent, le décalage ne vient pas des scores.

**Seuils du développement sur le test.** La part de NORMAL en zone rouge doit être comparée à la cible de 10 % et la part de PNEUMONIA en zone verte à celle de 2 %. Un écart important est la démonstration, chiffrée, qu'on ne peut pas transporter les seuils d'un centre à l'autre.

**Calibration locale.** La médiane dit ce qu'on obtient en moyenne ; l'intervalle à 95 % dit ce qui peut arriver avec un échantillon de réglage unique. Un intervalle large pour les PNEUMONIA en zone verte est normal : à 98 %, le seuil bas repose sur les quelques radios PNEUMONIA les moins bien notées. Le nombre de radios de réglage nécessaire se lit sur l'axe horizontal de `calibration_locale.png`. Ces résultats sont **optimistes** : les deux moitiés viennent du même centre.

**Prévalences.** La valeur prédictive de la zone rouge chute quand la pneumonie devient rare : même avec 10 % de fausses alarmes chez les NORMAL, une grande part des radios rouges seront normales si la prévalence est basse. À dire tel quel dans le mémoire : la zone rouge sert à *ordonner* la lecture, pas à annoncer une pneumonie.

**Alertes.** Deux questions : (1) les radios alertées sont-elles plus souvent mal classées ? (colonnes « pour 100 alertées / non alertées »), (2) que coûte la règle de sécurité ? (moins de radios rassurantes en zone verte, colonnes « avec / sans règle »). Une alerte qui se déclenche sur beaucoup de radios sans concentrer les erreurs use la confiance des utilisateurs : dans ce cas, à signaler.

**Limites à écrire dans le mémoire.** Un seul jeu de données et un seul centre ; test déjà consulté ; étiquettes de la base publique non revues par un radiologue ; moitiés du même centre ; alertes jamais testées sur des images d'un autre centre ; aucune étude auprès de cliniciens. L'application reste un prototype de recherche.

**Suite :** envoyer `a_envoyer_notebook_15.zip`. Lancer ensuite l'application (voir `application_clinique/LISEZMOI.md`) avec un échantillon de radios étiquetées pour la calibration locale.